The full-record EDA mostly confirms the plan and sharpens it. It also leaves a few risks that no existing number can settle, which is why the next SQL round matters.

## What the full record settles

- **The global VIIRS/Aqua ratio is stable from 2012 to 2025** (5.6 to 6.4), including through Aqua's post-2022 drift. A fit on 2012 to 2019 is therefore plausible to test forward on 2020 to 2025.
- **Terra/Aqua is steady at 2.0 to 2.1 for 2003 to 2011.** The Terra-to-Aqua step for Nov 2000 to Jul 2002 has a good, drift-free calibration window. It is the most reliable link in your chain.
- **Argentina's ratio (5.1) is below the global mean, in the lower group with the central African countries.** It still has a regime mix inside it that the national mean hides.
- **Suomi NPP/NOAA-20 is about 1:1**, with deviations explained by gap days. Stitching the VIIRS reference series is low risk.
- **VIIRS sees fire in about 21% more 1° cells than Aqua**, so some VIIRS signal has no MODIS counterpart. This puts a ceiling on any back-cast, which I'd flag in the pitch.

## What is still unknown, and why it matters

All of these remain unexamined in the data:

1. **Argentina's ratio in 2012 to 2018 by month and area.** All the by-country and by-month ratios cover 2019 to 2025 only, but your calibration window is 2012 to 2019. If Argentina's seasonal ratio shifts between the two periods, the forward test will fail for reasons unrelated to drift.
2. **Whether cell-days really stabilize the ratio.** This is the central hypothesis, and the EDA only tested raw counts and total FRP. Nothing has yet been computed on cell-days or distinct cells.
3. **Interannual stability of the ratio within Argentina.** The global annual ratio hides regional swings, and a big fire year (such as the 2020 Pantanal/Delta fires or 2022 Corrientes) could dominate.
4. **Day/night composition of the ratio.** Terra's daytime share fell from about 0.75 to 0.61, but Aqua's stayed at 0.94 to 0.96. A day-only versus all-hours comparison is needed.
5. **The 17.1 April** is still untraced, and could be a data artifact that contaminates fits.
6. **The 2012 to 2018 Suomi NPP vs Aqua relationship by country** was never examined, and it is the core of the calibration.

## Revised SQL plan: the five queries that matter most

These replace the broad set I proposed earlier. They assume the same `arg` temp table and use the full record. The first block gives you the decision-critical answers, and the rest is optional.

## 1. Setup

In [5]:
import os
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Find repo root (folder containing .git), then load credentials from .env.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
load_dotenv(root / ".env")

missing = [k for k in ("R2_KEY_ID", "R2_SECRET", "CF_ACCOUNT_ID") if not os.environ.get(k)]
assert not missing, f"Missing env vars: {missing}"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

# One colour per satellite, used in every plot. Dark2 is colour-blind safe.
SAT_COLOR = {"Terra": "#d95f02", "Aqua": "#1b9e77", "Suomi NPP": "#7570b3", "NOAA-20": "#e7298a"}
SATS = list(SAT_COLOR)

# Cost switch. True: cube from 2019, row-level scans limited to one month. False: full record.
QUICK = False
FIRST_YEAR = 2019 if QUICK else 2000
WINDOW_MONTH = 8  # August: peak season in Argentina, active in several other regions
WIN_TXT = f"August only" if QUICK else "full year"


def in_year(y: int) -> str:
    """SQL filter for one year (QUICK: one month). Partition pruning plus a date range that skips row groups."""
    if not QUICK:
        return f"year = {y}"
    m = WINDOW_MONTH
    return f"(year = {y} AND acq_date_utc >= DATE '{y}-{m:02d}-01' AND acq_date_utc < DATE '{y}-{m + 1:02d}-01')"

In [6]:
con = duckdb.connect()
con.sql("SET memory_limit = '3GB'")
con.sql(f"SET temp_directory = '{(root / 'tmp_ml').as_posix()}'")  # spill to disk; gitignored
con.sql(f"""
    CREATE SECRET r2 (
        TYPE r2,
        KEY_ID '{os.environ["R2_KEY_ID"]}',
        SECRET '{os.environ["R2_SECRET"]}',
        ACCOUNT_ID '{os.environ["CF_ACCOUNT_ID"]}'
    )
""")

BUCKET = "r2://firms-data"
SILVER = f"read_parquet('{BUCKET}/silver/detections/*/*/*.parquet', hive_partitioning = true)"
AGG = f"'{BUCKET}/analysis/agg_monthly.parquet'"
FIRE = "is_vegetation_fire AND confidence_class <> 'low'"


def q(sql: str) -> pd.DataFrame:
    """Run SQL, return a DataFrame. Only call on small (aggregated) results."""
    return con.sql(sql).df()

In [7]:
# arg: Argentina detections 2003-2025, all four satellites, materialised locally.
con.sql(f"""
    CREATE OR REPLACE TABLE arg AS
    SELECT satellite, year, local_date, latitude, longitude, lat10, lon10, frp,
           daynight, is_vegetation_fire, confidence_class
    FROM {SILVER}
    WHERE country = 'Argentina' AND year BETWEEN 2003 AND 2025
""")
print(q("SELECT satellite, count(*) AS n FROM arg GROUP BY 1 ORDER BY 1"))

# det: global view for query D (NOAA-20 / Aqua monthly ratio). Lazy, reads R2 when queried.
con.sql(f"""
    CREATE OR REPLACE VIEW det AS
    SELECT satellite, country, local_date, year, is_vegetation_fire, confidence_class
    FROM {SILVER}
    WHERE source IN ('modis', 'viirs_noaa20') AND year BETWEEN 2019 AND 2025
""")

   satellite        n
0       Aqua  1030024
1    NOAA-20  1728797
2  Suomi NPP  2889092
3      Terra   582949


CountBIGINT


### A. Is the ratio stable *within Argentina* across the calibration and test windows?

In [8]:
r = q(r"""
-- Monthly Suomi NPP / Aqua ratio, Argentina, 2012-2025, daytime only,
-- under four candidate units. The goal is to see which unit has a
-- stable ratio across months AND across the 2012-2019 vs 2020-2025 split.
WITH base AS (
  SELECT satellite, year, DATE_TRUNC('month', local_date) AS mon,
         lat10, lon10, local_date, frp
  FROM arg
  WHERE is_vegetation_fire AND confidence_class <> 'low'
    AND daynight = 'D'
    AND satellite IN ('Aqua','Suomi NPP')
    AND year BETWEEN 2012 AND 2025
),
agg AS (
  SELECT satellite, year, mon,
    COUNT(*)                                                AS n_det,
    COUNT(DISTINCT lat10 || '_' || lon10 || '_' || local_date) AS n_cell_days,
    COUNT(DISTINCT lat10 || '_' || lon10)                   AS n_cells,
    SUM(frp)                                                AS frp_sum
  FROM base GROUP BY ALL
)
SELECT a.year, MONTH(a.mon) AS m,
  v.n_det / NULLIF(a.n_det, 0)               AS r_counts,
  v.n_cell_days / NULLIF(a.n_cell_days, 0)   AS r_cell_days,
  v.n_cells / NULLIF(a.n_cells, 0)           AS r_cells,
  v.frp_sum / NULLIF(a.frp_sum, 0)           AS r_frp,
  a.n_cell_days AS aqua_cd, v.n_cell_days AS snpp_cd
FROM agg a JOIN agg v
  ON a.mon = v.mon AND a.satellite = 'Aqua' AND v.satellite = 'Suomi NPP'
ORDER BY a.year, m;
""")
r

,year,m,r_counts,r_cell_days,r_cells,r_frp,aqua_cd,snpp_cd
0,2012,1,1.304,0.932,0.963,0.239,1383,1289
1,2012,2,5.288,3.458,2.488,1.679,786,2718
2,2012,3,4.701,3.005,2.192,1.254,1161,3489
3,2012,4,4.948,3.817,3.321,1.279,377,1439
4,2012,5,4.082,3.436,2.640,1.425,761,2615
...,...,...,...,...,...,...,...,...
163,2025,8,3.010,2.143,1.590,0.813,4149,8891
164,2025,9,2.934,2.174,1.746,0.667,2679,5825
165,2025,10,3.372,2.463,2.070,0.874,1677,4131
166,2025,11,3.891,2.816,2.426,0.837,718,2022


Then summarize it as the coefficient of variation of each ratio, by window:

In [9]:
con.register("r", r)  # expose df to SQL
q(r"""
SELECT CASE WHEN year <= 2019 THEN '2012-2019' ELSE '2020-2025' END AS win,
       'counts' AS unit, AVG(r_counts) mean_r, STDDEV(r_counts)/AVG(r_counts) cv FROM r WHERE aqua_cd >= 30 GROUP BY 1
UNION ALL SELECT CASE WHEN year <= 2019 THEN '2012-2019' ELSE '2020-2025' END, 'cell_days', AVG(r_cell_days), STDDEV(r_cell_days)/AVG(r_cell_days) FROM r WHERE aqua_cd >= 30 GROUP BY 1
UNION ALL SELECT CASE WHEN year <= 2019 THEN '2012-2019' ELSE '2020-2025' END, 'cells', AVG(r_cells), STDDEV(r_cells)/AVG(r_cells) FROM r WHERE aqua_cd >= 30 GROUP BY 1
UNION ALL SELECT CASE WHEN year <= 2019 THEN '2012-2019' ELSE '2020-2025' END, 'frp', AVG(r_frp), STDDEV(r_frp)/AVG(r_frp) FROM r WHERE aqua_cd >= 30 GROUP BY 1;
""")

,win,unit,mean_r,cv
0,2020-2025,counts,4.392,0.244
1,2012-2019,counts,4.184,0.166
2,2012-2019,cell_days,2.966,0.185
3,2020-2025,cell_days,3.026,0.238
4,2020-2025,cells,2.397,0.270
5,2012-2019,cells,2.403,0.206
6,2012-2019,frp,1.068,0.252
7,2020-2025,frp,1.091,0.233


**What you're looking for:** the unit with the **lowest CV** and the **smallest shift in mean between windows**. This single table decides your primary quantity. The `aqua_cd >= 30` filter keeps near-empty winter months from inflating the variance, but also record how many months it drops, since a unit that only works in fire season is a problem for an all-year calendar.

### B. Where inside Argentina does the ratio differ? (the strata evidence)

In [10]:
q(r"""
-- Ratio by 1-degree cell, 2012-2019, cell-days unit. Export and map it.
WITH base AS (
  SELECT satellite, FLOOR(latitude) AS lat1, FLOOR(longitude) AS lon1,
         lat10, lon10, local_date,
         CASE WHEN MONTH(local_date) IN (12,1,2) THEN 'DJF'
              WHEN MONTH(local_date) IN (3,4,5)  THEN 'MAM'
              WHEN MONTH(local_date) IN (6,7,8)  THEN 'JJA' ELSE 'SON' END AS season
  FROM arg
  WHERE is_vegetation_fire AND confidence_class <> 'low' AND daynight = 'D'
    AND satellite IN ('Aqua','Suomi NPP') AND year BETWEEN 2012 AND 2019
)
SELECT lat1, lon1, season,
  COUNT(DISTINCT lat10||'_'||lon10||'_'||local_date) FILTER (WHERE satellite='Aqua')      AS aqua_cd,
  COUNT(DISTINCT lat10||'_'||lon10||'_'||local_date) FILTER (WHERE satellite='Suomi NPP') AS snpp_cd,
  COUNT(*) FILTER (WHERE satellite='Aqua')      AS aqua_n,
  COUNT(*) FILTER (WHERE satellite='Suomi NPP') AS snpp_n
FROM base GROUP BY ALL HAVING aqua_cd >= 5;
""")

,lat1,lon1,season,aqua_cd,snpp_cd,aqua_n,snpp_n
0,-37.000,-58.000,SON,40,86,49,177
1,-30.000,-63.000,SON,760,2375,1471,6637
2,-30.000,-58.000,SON,427,1332,671,2586
3,-27.000,-60.000,SON,513,2025,773,3850
4,-32.000,-63.000,DJF,46,67,61,98
...,...,...,...,...,...,...,...
836,-34.000,-60.000,JJA,281,702,445,1777
837,-39.000,-70.000,JJA,23,162,27,216
838,-28.000,-67.000,JJA,48,161,91,400
839,-30.000,-66.000,JJA,32,65,62,176


Map `snpp_cd / aqua_cd` per season. If it varies smoothly (north to south, or east to west), simple lat/lon-based strata may suffice. If it patches into distinct blocks, you have regimes, and the blocks suggest the strata.

### C. Detection recall: how much of Suomi NPP's signal does Aqua see?

In [11]:
q(r"""
-- Event-level coincidence at 0.1 degree cell-day, 2012-2019, daytime
WITH a AS (SELECT DISTINCT lat10, lon10, local_date FROM arg
           WHERE satellite='Aqua' AND is_vegetation_fire AND confidence_class<>'low'
             AND daynight='D' AND year BETWEEN 2012 AND 2019),
     v AS (SELECT DISTINCT lat10, lon10, local_date FROM arg
           WHERE satellite='Suomi NPP' AND is_vegetation_fire AND confidence_class<>'low'
             AND daynight='D' AND year BETWEEN 2012 AND 2019),
     both_cd AS (SELECT COUNT(*) n FROM a SEMI JOIN v USING (lat10, lon10, local_date))
SELECT (SELECT COUNT(*) FROM a) AS aqua_cd,
       (SELECT COUNT(*) FROM v) AS snpp_cd,
       (SELECT n FROM both_cd)  AS both_cd,
       (SELECT n FROM both_cd) * 1.0 / (SELECT COUNT(*) FROM a) AS frac_aqua_seen_by_viirs,
       (SELECT n FROM both_cd) * 1.0 / (SELECT COUNT(*) FROM v) AS frac_viirs_seen_by_aqua;
""")

,aqua_cd,snpp_cd,both_cd,frac_aqua_seen_by_viirs,frac_viirs_seen_by_aqua
0,132164,361746,90132,0.682,0.249


If `frac_aqua_seen_by_viirs` is high (say above 0.8), Aqua is nearly a subset of VIIRS and the problem reduces to estimating the extra VIIRS detections. If it is low, the two sensors are seeing different events on the same day, probably through overpass timing, and a regression on totals will be weaker than it looks. Note that the two satellites may pass at different times of day, so expect a lower number than a pure sensitivity comparison would give.

### D. Trace the 17.1 April, and look for others in Argentina

In [12]:
q(r"""
-- Global top-10 anomalous NOAA-20 / Aqua months
WITH m AS (
  SELECT satellite, DATE_TRUNC('month', local_date) mon, COUNT(*) n
  FROM det WHERE is_vegetation_fire AND confidence_class<>'low'
    AND satellite IN ('Aqua','NOAA-20') AND year BETWEEN 2019 AND 2025
  GROUP BY ALL
)
SELECT n.mon, n.n * 1.0 / a.n AS ratio, a.n aqua_n, n.n n20_n
FROM m a JOIN m n ON a.mon = n.mon AND a.satellite='Aqua' AND n.satellite='NOAA-20'
ORDER BY ratio DESC LIMIT 10;

-- If the top month is April of some year, find which countries drive it:
-- (replace the date range with the month you found)
SELECT country,
  COUNT(*) FILTER (WHERE satellite='NOAA-20') AS n20,
  COUNT(*) FILTER (WHERE satellite='Aqua')    AS aqua,
  COUNT(*) FILTER (WHERE satellite='NOAA-20') * 1.0 /
    NULLIF(COUNT(*) FILTER (WHERE satellite='Aqua'), 0) AS ratio
FROM det
WHERE is_vegetation_fire AND confidence_class<>'low'
  AND local_date >= DATE '2020-04-01' AND local_date < DATE '2020-05-01'  -- edit
GROUP BY country ORDER BY n20 DESC LIMIT 15;
""")

,country,n20,aqua,ratio
0,Russian_Federation,163379,25773,6.339
1,China,95206,7957,11.965
2,Myanmar,83995,11664,7.201
3,India,76275,6652,11.466
4,Venezuela,64582,8012,8.061
5,Mexico,56751,10028,5.659
6,Honduras,52687,6933,7.599
7,Democratic_Republic_of_the_Congo,48664,8147,5.973
8,Lao_PDR,40773,9625,4.236
9,Australia,35645,5307,6.717


### E. Day-only vs all-hours: does the daytime restriction matter in Argentina?

In [13]:
q(r"""
SELECT satellite, year,
  COUNT(*) AS n_all,
  COUNT(*) FILTER (WHERE daynight='D') AS n_day,
  COUNT(*) FILTER (WHERE daynight='D') * 1.0 / COUNT(*) AS day_share
FROM arg
WHERE is_vegetation_fire AND confidence_class<>'low'
  AND satellite IN ('Aqua','Terra','Suomi NPP','NOAA-20')
GROUP BY ALL ORDER BY satellite, year;
""")

,satellite,year,n_all,n_day,day_share
0,Aqua,2003,72997,67374,0.923
1,Aqua,2004,60937,56705,0.931
2,Aqua,2005,41999,39052,0.930
3,Aqua,2006,51148,47900,0.936
4,Aqua,2007,48465,45178,0.932
...,...,...,...,...,...
63,Terra,2021,18243,14654,0.803
64,Terra,2022,24943,17236,0.691
65,Terra,2023,16417,11653,0.710
66,Terra,2024,14131,9948,0.704


If the day share is steady for Aqua and Suomi NPP across 2012 to 2019 in Argentina, restricting to daytime is a safe simplification. If it is not, an all-hours fit carries a hidden diurnal bias.

## Two additional checks that are cheap and worth running

In [14]:
q(r"""
-- F. Terra / Aqua, Argentina, 2003-2011, daytime, by season and year.
SELECT year,
  CASE WHEN MONTH(local_date) IN (12,1,2) THEN 'DJF'
       WHEN MONTH(local_date) IN (3,4,5)  THEN 'MAM'
       WHEN MONTH(local_date) IN (6,7,8)  THEN 'JJA' ELSE 'SON' END AS season,
  COUNT(*) FILTER (WHERE satellite='Terra') AS terra_n,
  COUNT(*) FILTER (WHERE satellite='Aqua')  AS aqua_n,
  COUNT(*) FILTER (WHERE satellite='Terra') * 1.0
    / NULLIF(COUNT(*) FILTER (WHERE satellite='Aqua'), 0) AS terra_over_aqua
FROM arg
WHERE is_vegetation_fire AND confidence_class <> 'low' AND daynight = 'D'
  AND satellite IN ('Terra','Aqua') AND year BETWEEN 2003 AND 2011
GROUP BY ALL ORDER BY year, season;
""")

,year,season,terra_n,aqua_n,terra_over_aqua
0,2003,DJF,3848,6709,0.574
1,2003,JJA,9334,30376,0.307
2,2003,MAM,1525,4451,0.343
3,2003,SON,14508,25838,0.561
4,2004,DJF,4130,5786,0.714
5,2004,JJA,7840,24251,0.323
6,2004,MAM,2673,6291,0.425
7,2004,SON,11845,20377,0.581
8,2005,DJF,3685,4834,0.762
9,2005,JJA,4879,13857,0.352


In [15]:
q(r"""
-- G. Extreme-FRP detections in Argentina that could dominate FRP sums
SELECT satellite, year, frp, latitude, longitude, local_date
FROM arg WHERE is_vegetation_fire AND confidence_class <> 'low'
ORDER BY frp DESC LIMIT 30;
""")

,satellite,year,frp,latitude,longitude,local_date
0,Terra,2010,"13,294.500",-26.128,-63.812,2010-11-18
1,Terra,2010,"13,292.800",-26.127,-63.805,2010-11-18
2,Terra,2010,"10,680.100",-26.144,-63.806,2010-11-18
3,Aqua,2014,"10,647.900",-24.157,-63.910,2014-10-09
4,Terra,2010,"9,851.500",-26.145,-63.813,2010-11-18
5,Terra,2021,"9,399.600",-24.520,-61.103,2021-08-19
6,Terra,2021,"9,393.900",-24.526,-61.099,2021-08-19
7,Terra,2021,"9,322.700",-24.527,-61.070,2021-08-19
8,Aqua,2017,"8,887.700",-24.547,-60.675,2017-08-25
9,Aqua,2017,"8,473.900",-24.544,-60.672,2017-08-25


Query F tests the most trustworthy link in your chain on Argentine data specifically. Query G is a quick sanity check, because your EDA found single detections of up to 12,470 MW that would distort any FRP total.

## How to read the results: decision rules

| Result | Implication |
|---|---|
| A: cell-days has clearly lower CV and a smaller window shift than counts | Make cell-days the primary quantity |
| A: all units shift between 2012-2019 and 2020-2025 | Drift in Argentina is real, so calibrate on 2012-2019 only and report forward-test error honestly |
| B: ratio varies smoothly in space | Few strata, or a smooth spatial model |
| B: ratio varies in patches by season | Use ecoregion x season strata, with shrinkage |
| C: recall above ~0.8 | Aqua is mostly a subset, so use a detection-probability framing |
| C: recall below ~0.5 | Overpass-time mismatch dominates, so reconsider matching Aqua with Suomi NPP at all |
| F: Terra/Aqua stable by season in Argentina | The 2000 to 2002 extension is solid |
| E: day share unstable | Restrict to daytime for calibration |

One caution: do not tune several things at once on the full record. Pick the unit from A, then the strata from B, then evaluate the forward test once, so you don't overfit your choices to the test window.

In [ ]:
# Export outputs only (no code) to PDF. Save the notebook first (Ctrl+S) so the outputs are on disk.
# Needs: pip install "nbconvert[webpdf]" && playwright install chromium
import subprocess, sys

nb_path = root / "exploration" / "eda_firms_v2.ipynb"
subprocess.run(
    [sys.executable, "-m", "jupyter", "nbconvert", "--to", "webpdf",
     "--no-input", "--no-prompt",
     "--output", "eda_firms_v2_outputs", str(nb_path)],
    check=True,
)
print("wrote", nb_path.with_name("eda_firms_v2_outputs.pdf"))

wrote c:\repos\kubranauts-harmonization-hotspots\exploration\eda_firms_v2_outputs.html
